In [1]:
import os
import numpy as np
import cv2
import nibabel as nib
import tensorflow as tf

from tensorflow.keras.utils import Sequence
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score

2026-03-24 07:21:49.001148: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774336909.257301      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774336909.332760      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774336909.867718      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774336909.867761      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774336909.867765      55 computation_placer.cc:177] computation placer alr

In [2]:
!pip install nibabel -q

In [21]:
import os
import numpy as np
import cv2
import nibabel as nib
import tensorflow as tf
from tensorflow.keras.utils import Sequence
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score

# --- CONFIGURATION ---
IMG_SIZE = 224
BATCH_SIZE = 16
# Confirmed Path
base_path = "/kaggle/input/datasets/rksrank1/pancreatic-cancer/Task07_Pancreas"
image_path = os.path.join(base_path, "imagesTr")
mask_path = os.path.join(base_path, "labelsTr")

# 1. CLEAN FILE LOADING (Ignore hidden ._ files)
all_files = sorted([f for f in os.listdir(image_path) 
                    if (f.endswith(".nii") or f.endswith(".nii.gz")) 
                    and not f.startswith(".")])

train_files, test_files = train_test_split(all_files, test_size=0.2, random_state=42)
print(f"Total Patients: {len(all_files)} | Training: {len(train_files)} | Testing: {len(test_files)}")

# --- BALANCED GENERATOR ---
class BalancedCTGenerator(Sequence):
    def __init__(self, patient_files, batch_size=BATCH_SIZE, img_size=IMG_SIZE, is_training=True):
        self.patient_files = patient_files
        self.batch_size = batch_size
        self.img_size = img_size
        self.is_training = is_training
        self.samples = self._prepare_samples()

    def _prepare_samples(self):
        tumor_samples = []
        healthy_samples = []
        
        print(f"Indexing slices for {'Training' if self.is_training else 'Testing'}...")
        for file in self.patient_files:
            m_path = os.path.join(mask_path, file)
            mask_vol = nib.load(m_path).get_fdata()
            for i in range(mask_vol.shape[2]):
                if np.any(mask_vol[:, :, i] == 2): # Class 2 is tumor
                    tumor_samples.append((file, i, 1))
                else:
                    healthy_samples.append((file, i, 0))
        
        if self.is_training:
            # Under-sample healthy slices to create a 50/50 split
            np.random.shuffle(healthy_samples)
            healthy_samples = healthy_samples[:len(tumor_samples)]
            print(f"Balanced to {len(tumor_samples)} tumor and {len(healthy_samples)} healthy slices.")
        
        all_samples = tumor_samples + healthy_samples
        np.random.shuffle(all_samples)
        return all_samples

    def __len__(self):
        return int(np.ceil(len(self.samples) / self.batch_size))

    def __getitem__(self, idx):
        batch = self.samples[idx * self.batch_size : (idx + 1) * self.batch_size]
        X, y = [], []
        
        for file, slice_idx, label in batch:
            img_vol = nib.load(os.path.join(image_path, file)).get_fdata()
            slice_img = img_vol[:, :, slice_idx]
            
            # --- PANCREAS WINDOWING ---
            # CT values for soft tissue: Clip to [-100, 200] HU
            slice_img = np.clip(slice_img, -100, 200)
            slice_img = (slice_img + 100) / 300.0 # Normalize 0 to 1
            
            # Prepare for MobileNetV2 (RGB 224x224)
            slice_img = cv2.resize(slice_img, (self.img_size, self.img_size))
            slice_img = cv2.cvtColor((slice_img * 255).astype(np.uint8), cv2.COLOR_GRAY2RGB)
            
            X.append(slice_img / 255.0)
            y.append(label)
            
        return np.array(X), np.array(y)

# --- MODEL SETUP ---
train_gen = BalancedCTGenerator(train_files, is_training=True)
test_gen = BalancedCTGenerator(test_files, is_training=False)

base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(IMG_SIZE, IMG_SIZE, 3))
base_model.trainable = False # Freeze base first to stabilize

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(64, activation='relu')(x)
output = Dense(1, activation='sigmoid')(x)

model = Model(inputs=base_model.input, outputs=output)
model.compile(optimizer=Adam(learning_rate=1e-4), loss='binary_crossentropy', metrics=['accuracy', tf.keras.metrics.Recall()])

# --- TRAINING ---
print("\nStarting Training...")
model.fit(train_gen, validation_data=test_gen, epochs=5)

# --- EVALUATION (Slice-Level) ---
print("\nRunning Slice-Level Evaluation...")
y_true = []
y_pred = []

# We evaluate 5 batches of the test generator for a quick report
for i in range(min(10, len(test_gen))):
    batch_x, batch_y = test_gen[i]
    preds = model.predict(batch_x, verbose=0)
    y_true.extend(batch_y)
    y_pred.extend((preds > 0.5).astype(int).flatten())

print("\nClassification Report:")
print(classification_report(y_true, y_pred))
try:
    print("ROC-AUC:", roc_auc_score(y_true, y_pred))
except:
    print("ROC-AUC: Could not calculate (needs both classes in sample)")

Total Patients: 281 | Training: 224 | Testing: 57
Indexing slices for Training...
Balanced to 2067 tumor and 2067 healthy slices.
Indexing slices for Testing...


I0000 00:00:1774340024.763560      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1774340024.769441      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step

Starting Training...


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/5


...

158/259 ━━━━━━━━━━━━━━━━━━━━ 5:44 3s/step - accuracy: 0.6638 - loss: 0.6026 - recall: 0.7016

2026-03-24 08:24:22.325774: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-03-24 08:24:22.462174: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


259/259 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.6959 - loss: 0.5701 - recall: 0.7461

2026-03-24 08:34:14.179784: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-03-24 08:34:14.315186: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


259/259 ━━━━━━━━━━━━━━━━━━━━ 1173s 4s/step - accuracy: 0.6961 - loss: 0.5699 - recall: 0.7464 - val_accuracy: 0.6356 - val_loss: 0.6410 - val_recall: 0.8915
Epoch 2/5
259/259 ━━━━━━━━━━━━━━━━━━━━ 841s 3s/step - accuracy: 0.8091 - loss: 0.4325 - recall: 0.8886 - val_accuracy: 0.6732 - val_loss: 0.5958 - val_recall: 0.8809
Epoch 3/5
259/259 ━━━━━━━━━━━━━━━━━━━━ 747s 3s/step - accuracy: 0.8289 - loss: 0.4067 - recall: 0.9033 - val_accuracy: 0.7127 - val_loss: 0.5126 - val_recall: 0.8234
Epoch 4/5
259/259 ━━━━━━━━━━━━━━━━━━━━ 810s 3s/step - accuracy: 0.8466 - loss: 0.3738 - recall: 0.9331 - val_accuracy: 0.7544 - val_loss: 0.4396 - val_recall: 0.7638
Epoch 5/5
259/259 ━━━━━━━━━━━━━━━━━━━━ 769s 3s/step - accuracy: 0.8327 - loss: 0.3733 - recall: 0.9056 - val_accuracy: 0.7386 - val_loss: 0.4687 - val_recall: 0.8000

Running Slice-Level Evaluation...

Classification Report:
              precision    recall  f1-score   support

           0       0.97      0.77      0.86       146
           

In [20]:
def check_imbalance(files):
    total_slices = 0
    tumor_slices = 0
    
    for file in files:
        mask = nib.load(os.path.join(mask_path, file)).get_fdata()
        # Label 1 is Pancreas, Label 2 is Tumor. 
        # Your code checks for '2', which is correct for the tumor.
        slices_with_tumor = np.sum(np.any(mask == 2, axis=(0, 1)))
        
        total_slices += mask.shape[2]
        tumor_slices += slices_with_tumor
        
    print(f"Total Slices: {total_slices}")
    print(f"Tumor Slices (Class 1): {tumor_slices}")
    print(f"Healthy Slices (Class 0): {total_slices - tumor_slices}")
    print(f"Ratio: {100 * tumor_slices / total_slices:.2f}% Tumor")

check_imbalance(train_files)

Total Slices: 21532
Tumor Slices (Class 1): 2028
Healthy Slices (Class 0): 19504
Ratio: 9.42% Tumor


In [16]:
all_files = [f for f in os.listdir(image_path) if f.endswith(".nii")]
train_files, test_files = train_test_split(all_files, test_size=0.2, random_state=42)

train_gen = CTGenerator(train_files)
test_gen = CTGenerator(test_files, oversample_tumor=False)  # test: all slices

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input/pancreatic-cancer/Task07_Pancreas/imagesTr'

In [8]:

class CTGenerator(Sequence):
    def __init__(self, patient_files, image_path, mask_path, batch_size=8, img_size=224):
        self.patient_files = [f for f in patient_files if not f.startswith("._")]  # skip macOS hidden files
        self.image_path = image_path
        self.mask_path = mask_path
        self.batch_size = batch_size
        self.img_size = img_size
        self.samples = self._prepare_samples()

    def _prepare_samples(self):
        samples = []
        for file in self.patient_files:
            img_file = os.path.join(self.image_path, file)
            mask_file = os.path.join(self.mask_path, file)

            volume = nib.load(img_file).get_fdata()
            mask = nib.load(mask_file).get_fdata()

            for i in range(volume.shape[2]):
                slice_img = volume[:, :, i]
                slice_mask = mask[:, :, i]

                label = 1 if np.any(slice_mask == 2) else 0

                # Keep all tumor slices, very few normal slices
                if label == 1 or np.random.rand() < 0.05:
                    # ROI: only pancreas region
                    roi = (slice_mask > 0)
                    slice_img = slice_img * roi
                    samples.append((slice_img, label))

        np.random.shuffle(samples)
        return samples

    def __len__(self):
        return int(np.ceil(len(self.samples) / self.batch_size))

    def __getitem__(self, idx):
        batch = self.samples[idx * self.batch_size : (idx + 1) * self.batch_size]
        X, y = [], []

        for slice_img, label in batch:
            # normalize slice
            slice_img = (slice_img - np.min(slice_img)) / (np.max(slice_img) + 1e-8)
            slice_img = (slice_img * 255).astype(np.uint8)

            # resize and convert to RGB
            slice_img = cv2.resize(slice_img, (self.img_size, self.img_size))
            slice_img = cv2.cvtColor(slice_img, cv2.COLOR_GRAY2RGB)

            X.append(slice_img)
            y.append(label)

        return np.array(X) / 255.0, np.array(y)

In [10]:
train_gen = CTGenerator(train_files)
test_gen = CTGenerator(test_files)

print("Train samples:", len(train_gen.samples))

TypeError: CTGenerator.__init__() missing 2 required positional arguments: 'image_path' and 'mask_path'

In [ ]:
base_model = EfficientNetB0(
    weights='imagenet',
    include_top=False,
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)

x = base_model.output
x = GlobalAveragePooling2D()(x)
output = Dense(1, activation='sigmoid')(x)

model = Model(inputs=base_model.input, outputs=output)

model.compile(
    optimizer=Adam(1e-4),
    loss=focal_loss(),
    metrics=['accuracy']
)

In [ ]:
model.fit(
    train_gen,
    validation_data=test_gen,
    epochs=15
)

In [ ]:
def predict_patient(model, patient_file):
    volume = nib.load(os.path.join(image_path, patient_file)).get_fdata()
    mask = nib.load(os.path.join(mask_path, patient_file)).get_fdata()

    preds = []

    for i in range(volume.shape[2]):
        slice_img = volume[:, :, i]
        slice_mask = mask[:, :, i]

        roi = (slice_mask > 0)
        slice_img = slice_img * roi

        slice_img = (slice_img - np.min(slice_img)) / (np.max(slice_img) + 1e-8)
        slice_img = (slice_img * 255).astype(np.uint8)

        slice_img = cv2.resize(slice_img, (IMG_SIZE, IMG_SIZE))
        slice_img = cv2.cvtColor(slice_img, cv2.COLOR_GRAY2RGB)

        slice_img = np.expand_dims(slice_img/255.0, axis=0)

        pred = model.predict(slice_img, verbose=0)[0][0]
        preds.append(pred)

    # 🔥 LOWER threshold = better early detection
    return 1 if np.max(preds) > 0.3 else 0

In [ ]:
y_true, y_pred = [], []

for file in test_files:
    mask = nib.load(os.path.join(mask_path, file)).get_fdata()

    y_true.append(1 if np.any(mask == 2) else 0)
    y_pred.append(predict_patient(model, file))

print(classification_report(y_true, y_pred))

if len(set(y_true)) > 1:
    print("ROC-AUC:", roc_auc_score(y_true, y_pred))

In [ ]:
# Save model after training
model.save('/kaggle/working/pancreas_model.h5')

In [ ]:
import numpy as np
np.save('/kaggle/working/X.npy', X)
np.save('/kaggle/working/y.npy', y)